## The reference grid

The only campaign here that covers the space exhaustively, and therefore the only one that
can say what the eight searches beside it *missed*. It answers two questions a search
cannot: what did full coverage cost, and which of the two factors actually decides the
outcome.

**What to look for:** whether the doorway axis is monotone. It is not — and the width × dwell
table below says why, which reading the widths alone cannot.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import os
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from robovast.common.analysis import campaign_root, read_table


def note(message):
    """Say what is missing, in the output, instead of raising.

    These notebooks are also a deployment test: a traceback here is indistinguishable from
    the service failing to run notebooks at all, which is the thing under test.
    """
    print(f'[no data] {message}')


def factors(data_dir):
    """The doorway width and walker phase each configuration actually ran with.

    Read from _transient/configurations.yaml -- the RESOLVED overrides -- rather than from
    the .vast value lists or decoded out of the `grid-i-j` name. Those two are declarations;
    this is what was compiled into the world, and it is the only spelling that stays correct
    if the grid's levels are ever edited.

    Batch mode records nothing in the runs table's params_json (that carries SCENARIO
    parameters, and these vary the `sim:` channel), so this file is the only source.
    """
    path = os.path.join(campaign_root(data_dir), '_transient', 'configurations.yaml')
    if not os.path.isfile(path):
        return pd.DataFrame(columns=['config_name', 'gap_width', 'walker_dwell'])
    with open(path, encoding='utf-8') as handle:
        doc = yaml.safe_load(handle) or {}
    rows = []
    for cfg in doc.get('configs', []):
        over = ((cfg.get('sim') or {}).get('overrides') or {}).get('components', {})
        segments = (over.get('barrier') or {}).get('instances') or []
        # The gap is the gap, measured off the geometry: each segment runs from its centre
        # out by half its length, so the opening is twice the inner edge of the segment on
        # the positive side. Derived rather than looked up, so no room-width constant has to
        # be kept in step with the world.
        gap = None
        for seg in segments:
            pos, size = seg.get('pos'), seg.get('size')
            if pos and size and pos[1] > 0:
                gap = round(2 * (pos[1] - size[1] / 2), 3)
        rows.append({
            'config_name': cfg.get('name'),
            'gap_width': gap,
            'walker_dwell': (over.get('pedestrian') or {}).get('dwell'),
        })
    return pd.DataFrame(rows, columns=['config_name', 'gap_width', 'walker_dwell'])


runs = read_table(DATA_DIR, 'runs')
metrics = read_table(DATA_DIR, 'nav_metrics')
levels = factors(DATA_DIR)

if runs.empty:
    note('no runs recorded yet')
    cells_df = pd.DataFrame()
else:
    # Outcome from `runs` (every run has one) LEFT-joined to nav_metrics (only a run whose
    # postprocessing succeeded has one). An inner join here would silently drop exactly the
    # runs worth investigating, and shrink the denominator every rate below is divided by.
    keys = ['config_name', 'run_id']
    cells_df = runs[keys + ['passed']].merge(
        metrics[keys + ['collided', 'min_clearance', 'final_distance_to_goal',
                        'duration_s', 'recovery_count']],
        on=keys, how='left').merge(levels, on='config_name', how='left')
    cells_df['failed'] = cells_df['passed'] != 1

print(f'{len(cells_df)} run(s) over {cells_df["config_name"].nunique() if len(cells_df) else 0} cell(s)')
if len(cells_df) and cells_df['gap_width'].isna().any():
    note('some runs have no resolved factors - configurations.yaml did not name their config')


## The doorway axis, alone

Failures and collisions per width, over every walker phase. The two columns are what
separate the two failure mechanisms: a failure that is *not* a collision is the planner
refusing to attempt the gap.

In [ ]:
if not cells_df.empty:
    by_width = cells_df.groupby('gap_width').agg(
        runs=('failed', 'size'),
        failures=('failed', 'sum'),
        collisions=('collided', 'sum'),
    )
    by_width['failure_rate'] = by_width['failures'] / by_width['runs']
    # Of the runs that FAILED, how many did so by contact. 1.00 means failure and collision
    # are the same event at that width, so the geometry only decides whether the robot goes.
    by_width['collision_share_of_failures'] = (
        by_width['collisions'] / by_width['failures'].replace(0, pd.NA))
    display(by_width)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(by_width.index.astype(str), by_width['failures'], color='#c9611e', label='failures')
    ax.bar(by_width.index.astype(str), by_width['collisions'], color='#7a2f10',
           width=0.5, label='of which collisions')
    ax.set_xlabel('doorway width [m]'); ax.set_ylabel('runs')
    ax.set_title('Failures per doorway width')
    ax.legend(fontsize='small')
    plt.tight_layout(); plt.show()


## Both factors at once

The same runs, split by walker phase as well. This is the table the width axis alone cannot
be read without.

In [ ]:
if not cells_df.empty:
    grid = cells_df.pivot_table(index='gap_width', columns='walker_dwell',
                                values='failed', aggfunc='sum')
    counts = cells_df.pivot_table(index='gap_width', columns='walker_dwell',
                                  values='failed', aggfunc='size')
    print('failures per cell (of %s runs each):' % (
        int(counts.stack().mode().iloc[0]) if not counts.empty else 0))
    display(grid)

    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(grid.values, cmap='OrRd', aspect='auto', origin='lower')
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns)
    ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
    ax.set_xlabel("walker dwell [s]  (when it is in the way)")
    ax.set_ylabel('doorway width [m]')
    ax.set_title('Failures per cell')
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            value = grid.values[i, j]
            if pd.notna(value):
                ax.text(j, i, int(value), ha='center', va='center', fontsize=9)
    fig.colorbar(im, ax=ax, label='failures')
    plt.tight_layout(); plt.show()


In [ ]:
if not cells_df.empty:
    runs_spent = len(cells_df)
    cell_count = cells_df['config_name'].nunique()
    failures = int(cells_df['failed'].sum())

    # Which factor moves the outcome more: the spread of the per-level failure rate along
    # each axis, marginalised over the other. A range is the honest summary here because the
    # axis is NOT monotone -- a correlation would report ~0 for a factor that swings the
    # outcome from end to end and back.
    width_rate = cells_df.groupby('gap_width')['failed'].mean()
    dwell_rate = cells_df.groupby('walker_dwell')['failed'].mean()

    print(f"runs spent            : {runs_spent}  over {cell_count} cells")
    print(f"runs that failed      : {failures}  ({failures / runs_spent:.0%})")
    print()
    print("What this campaign is FOR -- the reference the searches are judged against:")
    print(f"  exhaustive coverage of this space cost {runs_spent} runs.")
    print(f"  A search is worth its complexity only if it finds the worst cell")
    print(f"  ({cells_df['failed'].groupby(cells_df['config_name']).mean().idxmax()}) for fewer.")
    print()
    print("  Which factor decides the outcome, marginally:")
    print(f"    doorway width : failure rate {width_rate.min():.0%} .. {width_rate.max():.0%}"
          f"  (span {width_rate.max() - width_rate.min():.0%})")
    print(f"    walker dwell  : failure rate {dwell_rate.min():.0%} .. {dwell_rate.max():.0%}"
          f"  (span {dwell_rate.max() - dwell_rate.min():.0%})")
    print()
    # Which factor is named here is READ OFF the spans above, never asserted. The point of
    # the block is that a view cannot claim a finding its own campaign does not support, and
    # a sentence naming a winner in advance is exactly that claim.
    spans = {'doorway width': width_rate.max() - width_rate.min(),
             'walker dwell': dwell_rate.max() - dwell_rate.min()}
    stronger = max(spans, key=spans.get)
    weaker = min(spans, key=spans.get)
    print(f"    the stronger factor here is {stronger} "
          f"({spans[stronger]:.0%} against {spans[weaker]:.0%})")
    print()
    monotone = width_rate.is_monotonic_decreasing or width_rate.is_monotonic_increasing
    print(f"  doorway axis monotone : {monotone}")
    if not monotone:
        print("    A wider doorway does not simply fail less, so the width axis cannot be")
        print("    read on its own -- the width x dwell table above is where the reason is.")
        print("    A handful of runs per width would report this ordering as a property of")
        print("    the geometry, when it is which mode those few runs happened to draw.")
